Problem Statement You are a data analyst at a movie streaming company. Your manager has asked you to build a small data pipeline using the TMDB API and analyze the collected data using Python and pandas.

Task 1 — Build the Pipeline Use the TMDB Discover Movies endpoint to pull data for at least 20 movies. Store the results in a pandas DataFrame and save it to a SQLite database table called movies.

In [ ]:
import os
import requests
import pandas as pd
import sqlite3
from google.colab import userdata

# Load API key securely from Colab secrets
#API_KEY = "YOUR_API_KEY_HERE"
api_key = userdata.get("TMDB_API_KEY")

if not api_key:
    raise ValueError("API_KEY not found. Did you add it in Tools → Secrets?")

BASE_URL = "https://api.themoviedb.org/3/discover/movie"

def fetch_movies(n=20):
    """Fetch at least n movies using TMDB Discover endpoint."""
    headers = {
        "Authorization": f"Bearer {api_key}",
        "accept": "application/json"
    }
    params = {
        "sort_by": "popularity.desc",
        "page": 1
    }
    response = requests.get(BASE_URL, headers=headers, params=params)
    if response.status_code != 200:
        raise Exception(f"API request failed: {response.status_code}")
    data = response.json()
    movies = data["results"][:n]  # take first n movies
    return pd.DataFrame(movies)

# Fetch movies
movies_df = fetch_movies(20)

# Convert list columns to strings for SQLite compatibility
# Check for columns that are objects and potentially contain lists
for col in movies_df.columns:
    if movies_df[col].map(type).eq(list).any():
        movies_df[col] = movies_df[col].astype(str)
print(movies_df.head())
# Save to SQLite
conn = sqlite3.connect("movies.db")
movies_df.to_sql("movies", conn, if_exists="replace", index=False)
conn.close()

   adult                     backdrop_path       genre_ids       id  \
0  False  /1fkuBPid72KGS6WmtkEXMftZtkE.jpg        [80, 18]   875828   
1  False  /2I1OFQJ0L9T0dpU6FobKFWV2PxX.jpg       [878, 12]   687163   
2  False  /sdZSjtGUTSN8B3al5o0f2WoQfQQ.jpg   [878, 12, 14]    83533   
3  False  /nHxWyy18SvAZ8jJeemtS8k1UNjM.jpg    [28, 80, 53]  1290821   
4  False  /hz7TdCrpLLt2Dz7S3PS2HG9rpAO.jpg  [27, 9648, 80]  1159559   

  original_language                    original_title  \
0                en  Peaky Blinders: The Immortal Man   
1                en                 Project Hail Mary   
2                en              Avatar: Fire and Ash   
3                en                           Shelter   
4                en                          Scream 7   

                                            overview  popularity  \
0  After his estranged son gets embroiled in a Na...    337.6486   
1  Science teacher Ryland Grace wakes up on a spa...    303.9914   
2  In the wake of the deva

# New Section

In [ ]:
for col in movies_df.columns:
    # Check if any element in the column is a list
    if movies_df[col].apply(type).apply(lambda x: x == list).any():
        # If so, convert the entire column to string type
        movies_df[col] = movies_df[col].astype(str)

Task 2 — Perform EDA Load the movies table back into a DataFrame and complete the following:

Display the first 5 rows
Show summary statistics for numeric columns
Count the number of movies per genre
Identify columns with missing values

In [ ]:
# Load back into DataFrame
conn = sqlite3.connect("movies.db")
movies_df = pd.read_sql("SELECT * FROM movies", conn)
conn.close()

# 1. Display first 5 rows
print("First 5 rows:")
print(movies_df.head())

# 2. Summary statistics for numeric columns
print("\nSummary statistics:")
print(movies_df.describe())

# 3. Count number of movies per genre
# Genres are stored as a list of dicts in TMDB response, so flatten first
genre_counts = {}
for genres in movies_df["genre_ids"]:
    for g in eval(genres) if isinstance(genres, str) else genres:
        genre_counts[g] = genre_counts.get(g, 0) + 1
print("\nMovies per genre_id:")
print(genre_counts)

# 4. Identify columns with missing values
print("\nMissing values per column:")
print(movies_df.isnull().sum())


First 5 rows:
   adult                     backdrop_path       genre_ids       id  \
0      0  /1fkuBPid72KGS6WmtkEXMftZtkE.jpg        [80, 18]   875828   
1      0  /2I1OFQJ0L9T0dpU6FobKFWV2PxX.jpg       [878, 12]   687163   
2      0  /sdZSjtGUTSN8B3al5o0f2WoQfQQ.jpg   [878, 12, 14]    83533   
3      0  /nHxWyy18SvAZ8jJeemtS8k1UNjM.jpg    [28, 80, 53]  1290821   
4      0  /hz7TdCrpLLt2Dz7S3PS2HG9rpAO.jpg  [27, 9648, 80]  1159559   

  original_language                    original_title  \
0                en  Peaky Blinders: The Immortal Man   
1                en                 Project Hail Mary   
2                en              Avatar: Fire and Ash   
3                en                           Shelter   
4                en                          Scream 7   

                                            overview  popularity  \
0  After his estranged son gets embroiled in a Na...    337.6486   
1  Science teacher Ryland Grace wakes up on a spa...    303.9914   
2  In the wa

In [ ]:
%%writefile README.md
# TMDB EDA Assignment

Exploratory Data Analysis of movies data pulled from the TMDB Discover Movies API, stored in SQLite, and analyzed with Pandas.

## Project Structure
- tmdb_eda.ipynb — Jupyter Notebook with pipeline + EDA
- README.md — Project description

## Usage
Replace `API_KEY` at the top of the notebook with your own TMDB key, then run all cells.


Writing README.md
